In [1]:
from openai import OpenAI

openai_client = OpenAI()

In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

In [3]:
from minsearch import AppendableIndex

In [4]:
index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [5]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [6]:
import json

RAG_INSTRUCTIONS = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

RAG_PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return RAG_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )

In [7]:
question = "How do I create a dahsbord in Evidently?"
search_results = search(question)
user_prompt = build_prompt(question, search_results)

In [8]:
messages = [
    {"role": "system", "content": RAG_INSTRUCTIONS},
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

In [9]:
print(response.output_text)

The provided context does not contain specific instructions on how to create a dashboard in Evidently.


In [10]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Use only facts from the knowledge base when answering.
IMPORTANT: f you cannot find the answer, inform the user.
"""

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [11]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
)

response.usage.input_tokens

63

In [12]:
messages = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

110

In [13]:
tool_call = response.output[0]
tool_call

ResponseFunctionToolCall(arguments='{"query":"create dashboard Evidently"}', call_id='call_cCAkQkyUZIkrf3oggKKEgSIY', name='search', type='function_call', id='fc_0a3767e2a83af3fa006acb50e2f0e087d188d84148a248c9d0', async_=None, caller=None, namespace=None, status='completed')

In [14]:
messages.append(tool_call)

In [15]:
tool_call.arguments

'{"query":"create dashboard Evidently"}'

In [16]:
arguments = json.loads(tool_call.arguments)
arguments

{'query': 'create dashboard Evidently'}

In [17]:
search_results = search(query='create dashboard in Evidently')

In [18]:
search_results = search(**arguments)
search_results[:1]

[{'start': 0,
  'content': "<Check>\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\n## What is a Dashboard?\n\nA Dashboard provides a clear view of your AI application performance. You can use it:\n\n- to track evaluation results across multiple experiments;\n- to track live production quality over time.\n\nEach Project has its own Dashboard. It's empty at first.\n\nTo populate it, you need to run an evaluation and **save at least one Report** to the Project. You can then choose values from Reports to plot.\n\n![](/images/dashboard_llm_light.png)\n\n## Dashboard Tabs\n\n<Check>\n  Multiple Tabs are available in **Evidently Cloud** and **Evidently Enterprise**.\n</Check>\n\nYou can logically organize Panels within the same Dashboard into different Tabs.\n\n![](/images/dashboard_llm_tabs.gif)\n\n## **Dashboard Panels**\n\nA Panel is a visual element in the Dashboard that displays specified values in a single widget. Panels ca

In [19]:
call_output = {
    "type": "function_call_output",
    "call_id": tool_call.call_id,
    "output": json.dumps(search_results),
}

In [20]:
messages.append(call_output)
messages

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nUse only facts from the knowledge base when answering.\nIMPORTANT: f you cannot find the answer, inform the user.\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard Evidently"}', call_id='call_cCAkQkyUZIkrf3oggKKEgSIY', name='search', type='function_call', id='fc_0a3767e2a83af3fa006acb50e2f0e087d188d84148a248c9d0', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_cCAkQkyUZIkrf3oggKKEgSIY',
  'output': '[{"start": 0, "content": "<Check>\\n  Dashboard is available in **Evidently OSS**, **Evidently Cloud** and **Evidently Enterprise**.\\n</Check>\\n\\n## What is a Dashboard?\\n\\nA Dashboard provides a clear view of your AI application performance. You can use it:\\n\\n- to track evaluation res

In [21]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
)

response.usage.input_tokens

3612

In [22]:
from typing import Literal
from pydantic import BaseModel, Field


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [23]:
response = openai_client.responses.parse(
    model='gpt-4o-mini',
    input=messages,
    tools=[search_tool],
    text_format=RAGResponse
)

response.usage.input_tokens

3836

In [24]:
rag_response = response.output_parsed
print(rag_response.answer)

To create a dashboard in Evidently, follow these steps:

### Step 1: Create a Project
1. **Sign up and log in** to your Evidently account, if you haven't done so already.
2. Create a **Project** where your dashboard will reside. Ensure you've run evaluations and saved at least one **Report** in the Project to have data to populate your dashboard.

### Step 2: Add a Dashboard Tab
1. **Enter "Edit" mode** on your Dashboard by clicking the top right corner.
2. Click the **plus sign** to add a Tab. You can either create a custom Tab or use a pre-built template, depending on your needs.
   - To create a custom Tab, select "empty" and name the Tab.

### Step 3: Add Panels
1. Still in **Edit mode**, use the **"Add Panel" button** to create visual elements (Panels) on your Dashboard.
2. You can create various kinds of Panels, such as counters or plots.
3. Follow the prompts to configure each Panel, selecting the Metrics you want to visualize from your Reports. You can filter by Tags if necessa

In [25]:
##Agentic Loop

In [26]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [27]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

IMPORTANT: When you explore the knowledge base, make at least 3 different
searchers to make sure you explore the topic well.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [28]:
question = "How do I create a dahsbord in Evidently?"

In [29]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

In [30]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

In [31]:
message_history.extend(response.output)

In [32]:
message_history

[{'role': 'system',
  'content': "\nYou're a documentation assistant. \n\nAnswer the user question using the documentation knowledge base\n\nIMPORTANT: When you explore the knowledge base, make at least 3 different\nsearchers to make sure you explore the topic well.\n\nUse only facts from the knowledge base when answering.\nIf you cannot find the answer, inform the user.\n\nOur knowledge base is entirely about Evidently, so you don't need to \ninclude the word 'evidently' in search results\n"},
 {'role': 'user', 'content': 'How do I create a dahsbord in Evidently?'},
 ResponseFunctionToolCall(arguments='{"query":"create dashboard"}', call_id='call_19LT6V5rqxQZMI8ptAeJ07ev', name='search', type='function_call', id='fc_087baa64c74ded58006acb51b6da7087d18249dfbcc089ff21', async_=None, caller=None, namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"dashboard features"}', call_id='call_VwaOurKw35Fz9ogc4wgOX38T', name='search', type='function_call', id='fc_08

In [33]:
for message in response.output:
    if message.type == 'function_call':
        print(f'executing {message.name}({message.arguments})...')
        tool_call_output = make_call(message)
        message_history.append(tool_call_output)

executing search({"query":"create dashboard"})...
executing search({"query":"dashboard features"})...
executing search({"query":"dashboard customization"})...


In [34]:
len(message_history)

8

In [35]:
response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=message_history,
    tools=[search_tool],
)

response.usage.input_tokens

12147

In [36]:
print(response.output_text)

To create a dashboard in Evidently, you can follow these steps:

### 1. Create or Connect to a Project
Make sure to connect to Evidently Cloud and create a project first. Dashboards rely on having reports in the project to visualize evaluation results.

### 2. Adding Tabs to the Dashboard
You can logically organize your dashboard using tabs. To add a new tab:
- **Through Python API**: Use the command:
  ```python
  project.dashboard.add_tab("Another Tab")
  ```
- **Through UI**: 
  - Enter "Edit" mode on the Dashboard (top right corner).
  - Click the plus sign to add a tab, select "empty" to create a custom tab, and enter its name.

### 3. Adding Panels
Panels are the visual elements in the dashboard that display specific metrics.

#### Method 1: Using Python API
You can add various types of panels (text, counters, plots, etc.) programmatically. 

**Example Code:**
```python
from evidently.sdk.models import PanelMetric
from evidently.sdk.panels import DashboardPanelPlot

# Add a text 

In [ ]:
## Agentic Loop

In [37]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [38]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=[search_tool],
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To assist you with creating a dashboard, I will first search for information specifically about the process of dashboard creation to gather relevant guidelines and best practices. This foundational knowledge should provide a clear starting point.

Let's proceed with the search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The initial search provided a variety of resources detailing how to create and manage dashboards, including adding panels and organizing tabs. The key steps involve using a user interface or a Python API to create and customize the dashboard and its components.

For further clarity, I will perform additional searches focusing on how to utilize the UI for dashboard creation and any specific features related to adding panels. This will ensure that I cover both code-based and no-code approaches, providing a comprehensive answer.

Let's proceed with two more targeted searches: one on using the UI 

In [39]:
#adding more toolS

In [40]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [41]:
def make_call(tool_call):
    arguments = json.loads(tool_call.arguments)
    name = tool_call.name

    if name == 'search':
        result = search(**arguments)
    elif name == 'add_entry':
        result = add_entry(**arguments)
    else: 
        result = 'not found tool "{name}"'

    return {
        "type": "function_call_output",
        "call_id": tool_call.call_id,
        "output": json.dumps(result),
    }

In [42]:
#make tools work together

In [43]:
message_history = [
    {"role": "system", "content": instructions},
    {"role": "user", "content": question}
]
tools = [search_tool, add_entry_tool]

iteration_number = 1

while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 1...
ASSISTANT: To address your question about creating a dashboard, I will first search for any documentation related to the process of dashboard creation. This is a key feature that users often seek guidance on, and understanding the basic steps involved is essential for new users.

Let's start with the first search.
executing search({"query":"create dashboard"})...

iteraration number 2...
ASSISTANT: The first search yielded helpful documentation on creating dashboards. It explains how you can add panels to your dashboard using either the user interface or a Python API. The sections cover various aspects, such as adding tabs, inserting panels, deleting them, and the importance of having reports as data sources for populating the dashboard.

Next, I’ll analyze these insights and perform two additional searches: focusing on the user interface method for adding panels and exploring the management and customization of dashboards. This will provide a more comprehensive

In [44]:
message_history.append(
    {"role": "user", "content": "add this content to our database"}
)

In [45]:
while True:
    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=message_history,
        tools=tools,
    )

    print(f'iteraration number {iteration_number}...') 
    message_history.extend(response.output)

    has_function_calls = False

    for message in response.output:
        if message.type == 'function_call':
            print(f'executing {message.name}({message.arguments})...')
            tool_call_output = make_call(message)
            message_history.append(tool_call_output)
            has_function_calls = True

        if message.type == 'message':
            text = message.content[0].text
            print('ASSISTANT:', text)

    iteration_number = iteration_number + 1
    print()

    if not has_function_calls:
        break

iteraration number 4...
executing add_entry({"filename":"docs/platform/dashboard_creation_overview.mdx","title":"Creating a Dashboard","description":"Steps to create a dashboard in Evidently.","content":"### Steps to Create a Dashboard\n\n1. **Setting Up the Project**:\n   - Make sure you’ve connected to Evidently Cloud and created a project. The dashboard is associated with this project.\n\n2. **Adding the Dashboard**:\n   - By default, every project starts with an empty dashboard. \n\n3. **Adding Tabs**:\n   - You can organize your panels into multiple tabs for better clarity. \n   - To add a new tab, enter \"Edit\" mode on the Dashboard and click the plus sign to add a new tab.\n\n4. **Adding Panels**:\n   - You can add various types of panels, including text panels, counters, pie charts, line plots, and bar plots.\n   - **Via the UI**:\n     - Enter \"Edit\" mode and click \"Add Panel\".\n     - Follow the prompts to configure your panel, including selecting metrics and defining ho

In [46]:
index.docs[-1]

{'start': 0,
 'content': '### Steps to Create a Dashboard\n\n1. **Setting Up the Project**:\n   - Make sure you’ve connected to Evidently Cloud and created a project. The dashboard is associated with this project.\n\n2. **Adding the Dashboard**:\n   - By default, every project starts with an empty dashboard. \n\n3. **Adding Tabs**:\n   - You can organize your panels into multiple tabs for better clarity. \n   - To add a new tab, enter "Edit" mode on the Dashboard and click the plus sign to add a new tab.\n\n4. **Adding Panels**:\n   - You can add various types of panels, including text panels, counters, pie charts, line plots, and bar plots.\n   - **Via the UI**:\n     - Enter "Edit" mode and click "Add Panel".\n     - Follow the prompts to configure your panel, including selecting metrics and defining how data is visualized.\n     - After setting up, click "Save" to place the panel in your selected tab.\n\n   - **Via Python API**:\n     - Use the following code snippets to add panels:

In [47]:
#The QA Loop

In [48]:
tools = [search_tool, add_entry_tool]

message_history = [
    {"role": "system", "content": instructions},
]

iteration_number = 1

# Q&A loop
while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    message_history.append({"role": "user", "content": user_prompt})

    # tool-call loop
    while True:
        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=message_history,
            tools=tools,
        )

        print(f'iteraration number {iteration_number}...') 
        message_history.extend(response.output)

        has_function_calls = False

        for message in response.output:
            if message.type == 'function_call':
                print(f'executing {message.name}({message.arguments})...')
                tool_call_output = make_call(message)
                message_history.append(tool_call_output)
                has_function_calls = True

            if message.type == 'message':
                text = message.content[0].text
                print('ASSISTANT:', text)

        iteration_number = iteration_number + 1
        print()

        if not has_function_calls:
            break

You: Hi what are we?


iteraration number 1...
executing search({"query":"what are we"})...

iteraration number 2...
ASSISTANT: Evidently is an open-source framework designed for evaluating, testing, and monitoring data and AI systems. It provides a lightweight Python library and a self-hosted platform, enabling teams to build and maintain reliable AI products, from simple predictive models to complex systems powered by large language models. The goal is to support teams in ensuring high performance in their AI-based applications.

Now I want to dig deeper into specific functionalities and features of Evidently to provide a more comprehensive understanding of what it offers. I will search for entries related to its features and highlight any relevant aspects that may further clarify the question about what Evidently is.
executing search({"query":"Evidently features"})...
executing search({"query":"Evidently platform overview"})...

iteraration number 3...
ASSISTANT: Evidently is an open-source framework that

You: stop


In [49]:
class Agent:

    def __init__(self, llm_client, model, instructions, tools):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.tools = tools

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == 'search':
            result = search(**arguments)
        elif name == 'add_entry':
            result = add_entry(**arguments)
        else:
            result = 'not found tool "{name}"'

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }   

    def loop(self, user_prompt, message_history=None):
        if not message_history:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f'iteraration number {iteration_number}...') 
            message_history.extend(response.output)

            has_function_calls = False

            for message in response.output:
                if message.type == 'function_call':
                    print(f'executing {message.name}({message.arguments})...')
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                if message.type == 'message':
                    text = message.content[0].text
                    print('ASSISTANT:', text)

            iteration_number = iteration_number + 1
            print()

            if not has_function_calls:
                break

        return message_history        

    def qna(self):
        message_history = [
            {"role": "system", "content": instructions},
        ]

        iteration_number = 1

        # Q&A loop
        while True:
            user_prompt = input('You:')
            if user_prompt.lower().strip() == 'stop':
                break

            message_history = self.loop(user_prompt, message_history)

In [50]:
agent = Agent(
    llm_client=OpenAI(),
    model='gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [51]:
messages = agent.loop('evidently adshboards')

iteraration number 0...
ASSISTANT: To assist you with "evidently dashboards," I’ll start by searching for information specifically on dashboards first. This will help me understand the basic features, functionalities, and any relevant documentation regarding how dashboards operate within the platform.

Let's proceed with the search for "dashboards."
executing search({"query":"dashboards"})...

iteraration number 1...
ASSISTANT: The first search yielded various results related to dashboards, particularly focusing on adding and managing panels in the dashboard interface. Key points include how to create tabs, add panels through both a user interface and Python API, and examples of specific panel types such as distribution and plots.

Next, it’s important to analyze how these panels and dashboard functionalities interact and might touch upon specific user scenarios or capabilities, such as monitoring setups or specific configurations. Therefore, I will search for "adding dashboard panels"

In [ ]:
agent.qna()

You: what was implemented here


iteraration number 0...
ASSISTANT: To provide a clear answer about what was implemented, I will conduct a search for "implementations" in our documentation. This should give me insights into specific features or functionalities that have been added or modified. Understanding the details of implementations can help clarify the changes or enhancements made in a particular context.

Let's proceed with the search now.
executing search({"query":"implementations"})...

iteraration number 1...
ASSISTANT: The search for "implementations" returned relevant documentation about self-hosting the UI service and using the `fsspec` library to connect with data stores. Specifically, it details how snapshots can be read from remote data stores, such as an S3 bucket, and the environment variables required for configuration. This indicates that significant functionalities related to data handling and service deployment have been implemented.

Given this context, it would be valuable to delve deeper into 